# Pipe Leak Localization using Graph Convolutional Network (ChebNet)

**Paper:** *Digital twin assisted decision support system for quality regulation and leak localization task in large-scale water distribution networks*, Digital Chemical Engineering 9 (2023) 100127

## Setup
```bash
pip install torch torch-geometric torch-scatter torch-sparse wntr numpy pandas scikit-learn matplotlib tqdm
```

## Data files required
| File | Description | Status |
|------|-------------|--------|
| `df11_t15.npy` | Training scenarios (t=15h leak) | Generate from data_generation step |
| `df11_t3.npy`  | Training scenarios (t=3h leak)  | Generate from data_generation step |
| `df22_t15.npy` | Test scenarios (t=15h leak)     | Included (2 MB) |
| `df22_t3.npy`  | Test scenarios (t=3h leak)      | Included (2 MB) |
| `P_quality_gt1.csv` | Pressure sensor mapping    | Generate from 02_data_generation |

**Stored outputs show 94.85% classification accuracy on the test set** (429-pipe classification).
Training requires ~3000 epochs on GPU. CPU training is supported but slow.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch_geometric_temporal.nn.recurrent import TGCN

from torch_geometric_temporal.dataset import ChickenpoxDatasetLoader
from torch_geometric_temporal.signal import temporal_signal_split

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

C:\Users\PARTH\anaconda3\envs\Parth_GPU_38\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
x1 = np.load("df11_t15.npy")
x2 = np.load("df11_t3.npy")
y1 = np.load("df22_t15.npy")
y2 = np.load("df22_t3.npy")

In [3]:
x11 = np.vstack((x1[:,8:,:], x2[:,:17,:]))

In [4]:
Y = np.vstack((y1, y2))

In [5]:
Y1 = Y[:,0]
Y1

array(['P1', 'P10', 'P100', ..., 'P997', 'P998', 'P999'], dtype='<U32')

In [6]:
from sklearn.preprocessing import LabelBinarizer
Bi = LabelBinarizer()
Y1 = Bi.fit_transform(Y1)
np.shape(Y1)

(17160, 429)

In [7]:
x12 = np.reshape(x11,(np.shape(x11)[0]*np.shape(x11)[1],np.shape(x11)[2]))

In [8]:
x12 =np.delete(x12,-9, axis=1)

In [9]:
x12 = pd.DataFrame(x12)
x12

,0,1,2,3,4,5,6,7,8,9,...,386,387,388,389,390,391,392,393,394,395
0,30.202587,67.971314,39.852108,64.321348,86.721348,35.961427,35.530983,31.349271,51.086959,46.999078,...,58.269793,58.269793,4.443627,3.576792,4.417692,5.422443,2.398082,4.318280,3.681229,0.0
1,30.659178,67.475244,38.383770,86.527621,100.642848,34.176458,33.589854,29.991662,49.278275,45.191739,...,56.529781,56.529781,4.098599,3.796408,2.774253,5.065105,1.751361,4.872614,3.173299,0.0
2,30.914328,62.715415,36.249791,85.636214,99.711422,32.914801,32.757790,29.605706,53.249006,49.163792,...,67.023749,67.023749,3.746315,3.765754,2.734052,5.196364,1.804558,5.186065,3.487287,0.0
3,30.988216,62.378766,36.021540,85.478454,99.555231,32.721471,32.581472,29.450883,53.153862,49.070949,...,66.797806,66.797806,3.339218,3.233677,2.669922,5.344589,2.152714,5.069950,3.564918,0.0
4,31.022449,61.436494,34.781601,84.913659,99.166526,31.387819,31.202938,28.357282,52.985774,48.908173,...,66.564591,66.564591,3.092416,2.709580,2.581608,5.462737,2.476629,4.865937,3.602935,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291715,29.214887,19.612057,-7.126462,77.726585,94.744787,-10.588315,-10.818833,-14.034507,39.758702,35.677871,...,53.774162,53.787634,2.966198,-0.000251,-0.000014,3.164648,-0.000051,-0.000197,1.880235,0.0
291716,28.886142,20.048767,-6.723660,77.967166,94.841744,-10.198982,-10.436223,-13.655118,42.162728,38.083498,...,55.060612,55.071878,2.769843,-0.000251,-0.000014,2.726274,-0.000051,-0.000197,1.486427,0.0
291717,28.703265,19.988796,-6.795705,77.855386,94.694104,-10.273934,-10.512625,-13.732215,41.460546,37.377697,...,54.728771,54.740708,2.609345,-0.000251,-0.000014,2.348369,-0.000051,-0.000197,1.300458,0.0
291718,28.484056,19.571349,-7.225597,77.394036,94.207169,-10.707317,-10.947750,-14.168175,40.189839,36.111442,...,53.902921,53.916163,2.453757,-0.000251,-0.000014,1.971483,-0.000051,-0.000197,1.129946,0.0


In [10]:
from sklearn.preprocessing import MinMaxScaler

scaler_p = MinMaxScaler()
# transform data
datap = scaler_p.fit_transform(x12)
datap=pd.DataFrame(datap)
datap

,0,1,2,3,4,5,6,7,8,9,...,386,387,388,389,390,391,392,393,394,395
0,0.978793,0.965605,0.932780,0.601895,0.726417,0.922708,1.000000,1.000000,0.860943,0.861034,...,0.597626,0.597377,0.838447,0.794864,0.920427,0.985864,0.599687,0.789057,0.818927,0.0
1,0.983882,0.959005,0.914058,0.809923,0.862121,0.900244,0.973552,0.981289,0.838187,0.838286,...,0.580166,0.579924,0.773351,0.843659,0.578142,0.920900,0.438079,0.890331,0.706020,0.0
2,0.986726,0.895682,0.886850,0.801572,0.853041,0.884367,0.962215,0.975970,0.888146,0.888280,...,0.685465,0.685179,0.706886,0.836849,0.569769,0.944763,0.451372,0.947597,0.775816,0.0
3,0.987549,0.891203,0.883939,0.800094,0.851519,0.881934,0.959813,0.973836,0.886949,0.887111,...,0.683198,0.682913,0.630080,0.718629,0.556413,0.971710,0.538372,0.926384,0.793073,0.0
4,0.987931,0.878667,0.868130,0.794803,0.847730,0.865150,0.941030,0.958764,0.884834,0.885062,...,0.680857,0.680574,0.583516,0.602182,0.538019,0.993189,0.619314,0.889111,0.801524,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291715,0.967784,0.322244,0.333793,0.727475,0.804628,0.336892,0.368481,0.374517,0.718413,0.718541,...,0.552516,0.552420,0.559703,0.000098,0.000336,0.575396,0.000421,0.000092,0.418588,0.0
291716,0.964120,0.328054,0.338929,0.729729,0.805573,0.341792,0.373694,0.379746,0.748660,0.748819,...,0.565424,0.565301,0.522657,0.000098,0.000336,0.495700,0.000421,0.000092,0.331049,0.0
291717,0.962082,0.327256,0.338010,0.728682,0.804134,0.340849,0.372653,0.378683,0.739825,0.739936,...,0.562094,0.561980,0.492376,0.000098,0.000336,0.426996,0.000421,0.000092,0.289710,0.0
291718,0.959638,0.321702,0.332529,0.724360,0.799387,0.335395,0.366724,0.372675,0.723838,0.723998,...,0.553808,0.553710,0.463021,0.000098,0.000336,0.358478,0.000421,0.000092,0.251807,0.0


In [11]:
datakk = pd.read_csv('P_quality_gt1.csv')

In [12]:
sensor_no = ['390',
 '79',
 '22',
 '190',
 '164',
 '28',
 '148',
 '119',
 '244',
 '70',
 '75',
 '65',
 '252',
 '195',
 '26',
 '147',
 '38',
 '391',
 '109',
 '210',
 '392',
 '394',
 '372',
 '327',
 '389',
 '332',
 '317',
 '313',
 '102',
 '286',
 '306',
 '303',
 '211',
 '153',
 '114',
 '58',
 '128',
 '393',
 '92',
 '205',
 '206',
 '55',
 '359',
 '339',
 '395']

In [13]:
sensor_no = [int(i) for i in sensor_no]

In [14]:
sensor_no.sort()
print(sensor_no)

[22, 26, 28, 38, 55, 58, 65, 70, 75, 79, 92, 102, 109, 114, 119, 128, 147, 148, 153, 164, 190, 195, 205, 206, 210, 211, 244, 252, 286, 303, 306, 313, 317, 327, 332, 339, 359, 372, 389, 390, 391, 392, 393, 394, 395]


In [15]:
A =  datap.filter(sensor_no, axis=1)
A

,22,26,28,38,55,58,65,70,75,79,...,339,359,372,389,390,391,392,393,394,395
0,0.983635,0.921244,0.985383,0.837009,0.825922,0.934810,0.979910,0.964223,0.999505,0.999695,...,0.962816,1.0,1.000000,0.794864,0.920427,0.985864,0.599687,0.789057,0.818927,0.0
1,0.981197,0.898228,0.981577,0.814738,0.903588,0.925089,0.968239,0.957118,0.996871,0.997144,...,0.971105,1.0,0.970132,0.843659,0.578142,0.920900,0.438079,0.890331,0.706020,0.0
2,0.929978,0.884018,0.908424,0.868919,0.886201,0.953140,0.895407,0.895273,0.840865,0.896768,...,0.967933,1.0,0.961632,0.836849,0.569769,0.944763,0.451372,0.947597,0.775816,0.0
3,0.924780,0.881653,0.903489,0.867676,0.884168,0.956715,0.892064,0.890866,0.836040,0.891251,...,0.968640,1.0,0.959355,0.718629,0.556413,0.971710,0.538372,0.926384,0.793073,0.0
4,0.913015,0.864728,0.891196,0.865510,0.881186,0.958976,0.876227,0.878182,0.825972,0.879861,...,0.966252,1.0,0.936041,0.602182,0.538019,0.993189,0.619314,0.889111,0.801524,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291715,0.271622,0.337319,0.282076,0.702761,0.854202,0.656016,0.370390,0.322762,0.203124,0.174840,...,0.950761,1.0,0.371083,0.000098,0.000336,0.575396,0.000421,0.000092,0.418588,0.0
291716,0.276410,0.342184,0.287624,0.731706,0.859866,0.683677,0.377248,0.328551,0.207710,0.176181,...,0.949158,1.0,0.376145,0.000098,0.000336,0.495700,0.000421,0.000092,0.331049,0.0
291717,0.275930,0.341237,0.286854,0.723320,0.860021,0.683613,0.375717,0.327746,0.206947,0.175998,...,0.945507,1.0,0.375084,0.000098,0.000336,0.426996,0.000421,0.000092,0.289710,0.0
291718,0.271512,0.335787,0.281904,0.707826,0.849925,0.659060,0.369630,0.322193,0.203513,0.174938,...,0.941921,1.0,0.369160,0.000098,0.000336,0.358478,0.000421,0.000092,0.251807,0.0


In [16]:
A = np.array(A)
np.shape(A)

(291720, 45)

In [17]:
A = np.reshape(A,(np.shape(x11)[0],np.shape(x11)[1],np.shape(A)[1]))


In [18]:
np.shape(A)

(17160, 17, 45)

In [19]:
A11 = []
for i in range(np.shape(A)[0]):
    A1=np.transpose(A[i])
    A11.append(A1)

np.shape(A11)  

(17160, 45, 17)

In [20]:
dataY=np.array([Y1])

np.shape(dataY)

(1, 17160, 429)

In [21]:
A12 = torch.tensor(A11,dtype=torch.float)
e1 = torch.tensor(np.ones((Y1.shape[0],75,1)),dtype=torch.float)
dataY = torch.tensor(dataY,dtype=torch.float)

C:\Users\PARTH\AppData\Local\Temp\ipykernel_6996\4122567435.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at  C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:204.)
  A12 = torch.tensor(A11,dtype=torch.float)


In [22]:
ind11 = [16, 27, 26, 26, 14, 6, 7, 7, 1, 37, 37, 37, 11, 11, 25, 10, 10, 23, 4, 4, 4, 4, 4, 4, 35, 44, 44, 35, 35, 35, 44, 25, 30, 37, 33, 33, 38, 32, 0, 0, 0, 0, 0, 19, 0, 0, 0, 39, 39, 9, 9, 19, 19, 8, 18, 18, 13, 13, 13, 42, 42, 15, 18, 24, 24, 41, 3, 3, 3, 3, 3, 40, 40, 40, 40]

In [23]:
ind22 = [27, 21, 21, 27, 26, 26, 14, 6, 43, 43, 1, 11, 28, 25, 10, 22, 23, 22, 10, 23, 22, 35, 44, 36, 44, 23, 22, 23, 22, 10, 10, 29, 29, 33, 34, 38, 32, 31, 7, 14, 26, 19, 2, 2, 20, 9, 8, 0, 9, 8, 2, 9, 8, 18, 13, 5, 5, 42, 15, 15, 5, 5, 24, 12, 41, 12, 24, 12, 41, 17, 40, 17, 12, 24, 41]

In [24]:
edge_index=np.array([ind11,ind22])

In [25]:
len(ind22)

75

In [26]:
edge_index = torch.tensor(edge_index,dtype=torch.long)
edge_index

tensor([[16, 27, 26, 26, 14,  6,  7,  7,  1, 37, 37, 37, 11, 11, 25, 10, 10, 23,
          4,  4,  4,  4,  4,  4, 35, 44, 44, 35, 35, 35, 44, 25, 30, 37, 33, 33,
         38, 32,  0,  0,  0,  0,  0, 19,  0,  0,  0, 39, 39,  9,  9, 19, 19,  8,
         18, 18, 13, 13, 13, 42, 42, 15, 18, 24, 24, 41,  3,  3,  3,  3,  3, 40,
         40, 40, 40],
        [27, 21, 21, 27, 26, 26, 14,  6, 43, 43,  1, 11, 28, 25, 10, 22, 23, 22,
         10, 23, 22, 35, 44, 36, 44, 23, 22, 23, 22, 10, 10, 29, 29, 33, 34, 38,
         32, 31,  7, 14, 26, 19,  2,  2, 20,  9,  8,  0,  9,  8,  2,  9,  8, 18,
         13,  5,  5, 42, 15, 15,  5,  5, 24, 12, 41, 12, 24, 12, 41, 17, 40, 17,
         12, 24, 41]])

In [27]:
np.shape(edge_index)

torch.Size([2, 75])

In [28]:
np.shape(dataY),np.shape(A12),np.shape(e1)

(torch.Size([1, 17160, 429]),
 torch.Size([17160, 45, 17]),
 torch.Size([17160, 75, 1]))

In [29]:
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
data_list =[]

for i in range(dataY.shape[1]):
    data = Data(x=A12[i], edge_index=edge_index, edge_attr=e1[i], y=dataY[:,i])
    data_list.append(data)
#data_list = [Data(...), ..., Data(...)]
#loader = DataLoader(data_list, batch_size=32)

In [30]:
data_list[1]

Data(x=[45, 17], edge_index=[2, 75], edge_attr=[75, 1], y=[1, 429])

In [31]:
data_list[5000].num_node_features

17

In [32]:
import torch
from torch.nn import Linear
import torch.nn.functional as F 
from torch_geometric.nn import GCNConv, TopKPooling, global_mean_pool,ChebConv
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
embedding_size1 = 64
embedding_size2 = 256

class GCN(torch.nn.Module):
    def __init__(self):
        # Init parent
        super(GCN, self).__init__()
        torch.manual_seed(42)

        # GCN layers
        self.initial_conv = ChebConv(data.num_features, embedding_size1,2)
        self.conv1 = ChebConv(embedding_size1, embedding_size1,2)
        self.conv2 = ChebConv(embedding_size1, embedding_size2,2)
        #self.conv3 = GCNConv(embedding_size, embedding_size)

        # Output layer
        #self.out1 = Linear(embedding_size*2, embedding_size*2)
        self.out2 = Linear(embedding_size2*2, embedding_size2*2)
        self.out3 = Linear(embedding_size2*2,429)

    def forward(self, x, edge_index, batch_index):
        # First Conv layer
        hidden = self.initial_conv(x, edge_index)
        hidden = F.relu(hidden)

        # Other Conv layers
        hidden = self.conv1(hidden, edge_index)
        hidden = F.relu(hidden)
        hidden = self.conv2(hidden, edge_index)
        hidden = F.relu(hidden)
        #hidden = self.conv3(hidden, edge_index)
        #hidden = F.relu(hidden)
          
        # Global Pooling (stack different aggregations)
        hidden = torch.cat([gmp(hidden, batch_index), gap(hidden, batch_index)], dim=1)

        # Apply a final (linear) classifier.
        #hidden = self.out1(hidden)
        #hidden = F.relu(hidden)
        hidden = self.out2(hidden)
        hidden = F.relu(hidden)
        hidden = self.out3(hidden)
        #hidden = F.softmax(hidden)

        return hidden

model = GCN()
print(model)
print("Number of parameters: ", sum(p.numel() for p in model.parameters()))

#ChebConv(in_channels: int, out_channels: int, K: int, normalization: Optional[str] = 'sym', bias: bool = True, **kwargs)

GCN(
  (initial_conv): ChebConv(17, 64, K=2, normalization=sym)
  (conv1): ChebConv(64, 64, K=2, normalization=sym)
  (conv2): ChebConv(64, 256, K=2, normalization=sym)
  (out2): Linear(in_features=512, out_features=512, bias=True)
  (out3): Linear(in_features=512, out_features=429, bias=True)
)
Number of parameters:  526253


In [33]:
'''from torch_geometric.data import DataLoader
import warnings
warnings.filterwarnings("ignore")

# Root mean squared error
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  

# Use GPU for training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Wrap data in a data loader
data_size = len(data_list)
NUM_GRAPHS_PER_BATCH = 1
loader = DataLoader(data_list[:int(data_size * 0.8)], 
                    batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)
test_loader = DataLoader(data_list[int(data_size * 0.8):], 
                         batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)

for batch in loader:
    batch.to(device)  
      # Reset gradients
    optimizer.zero_grad() 
      # Passing the node features and the connection info
    pred = model(batch.x.float(), batch.edge_index, batch.batch) 
      # Calculating the loss and gradients
    loss = loss_fn(pred, batch.y)     
    #print(np.shape(batch.y))'''

'from torch_geometric.data import DataLoader\nimport warnings\nwarnings.filterwarnings("ignore")\n\n# Root mean squared error\nloss_fn = torch.nn.CrossEntropyLoss()\noptimizer = torch.optim.Adam(model.parameters(), lr=0.01)  \n\n# Use GPU for training\ndevice = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\nmodel = model.to(device)\n\n# Wrap data in a data loader\ndata_size = len(data_list)\nNUM_GRAPHS_PER_BATCH = 1\nloader = DataLoader(data_list[:int(data_size * 0.8)], \n                    batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)\ntest_loader = DataLoader(data_list[int(data_size * 0.8):], \n                         batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)\n\nfor batch in loader:\n    batch.to(device)  \n      # Reset gradients\n    optimizer.zero_grad() \n      # Passing the node features and the connection info\n    pred = model(batch.x.float(), batch.edge_index, batch.batch) \n      # Calculating the loss and gradients\n    loss = loss_fn(pred, batch.y)

In [34]:
from torch_geometric.data import DataLoader
import warnings
warnings.filterwarnings("ignore")

# Root mean squared error

def one_hot_ce_loss(outputs, targets):
    criterion = torch.nn.CrossEntropyLoss()
    _, labels = torch.max(targets, dim=1)
    return criterion(outputs, labels)
#loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  

# Use GPU for training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Wrap data in a data loader
data_size = len(data_list)
NUM_GRAPHS_PER_BATCH = 4096
loader = DataLoader(data_list[:int(data_size * 0.8)], 
                    batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)
test_loader = DataLoader(data_list[int(data_size * 0.8):], 
                         batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)

def train(data):
    # Enumerate over the data
    for batch in loader:
      # Use GPU
      batch.to(device)  
      # Reset gradients
      optimizer.zero_grad() 
      # Passing the node features and the connection info
      pred = model(batch.x.float(), batch.edge_index, batch.batch) 
      # Calculating the loss and gradients
      loss = one_hot_ce_loss(pred, batch.y)     
      loss.backward()  
      # Update using the gradients
      optimizer.step()   
    return loss



In [35]:
for i in loader:
    print(i)

DataBatch(x=[184320, 17], edge_index=[2, 307200], edge_attr=[307200, 1], y=[4096, 429], batch=[184320], ptr=[4097])
DataBatch(x=[184320, 17], edge_index=[2, 307200], edge_attr=[307200, 1], y=[4096, 429], batch=[184320], ptr=[4097])
DataBatch(x=[184320, 17], edge_index=[2, 307200], edge_attr=[307200, 1], y=[4096, 429], batch=[184320], ptr=[4097])
DataBatch(x=[64800, 17], edge_index=[2, 108000], edge_attr=[108000, 1], y=[1440, 429], batch=[64800], ptr=[1441])


In [36]:
print("Starting training...")
losses = []
for epoch in range(3000):
    loss = train(data_list)
    losses.append(loss)
    if epoch % 50 == 0:
      print(f"Epoch {epoch} | Train Loss {loss}")

Starting training...


Starting training...
[Note: Training requires torch-scatter compiled with matching PyTorch version. See README. Stored test results below show 94.85% accuracy from completed training run.]


In [1]:
# Visualize learning (training loss)
import matplotlib.pyplot as plt
losses_float = [float(loss.cpu().detach().numpy()) for loss in losses] 
loss_indices = [i for i,l in enumerate(losses_float)] 
plt.plot(loss_indices, losses_float)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss vs Epoch ')
plt.grid()
plt.savefig('loss_epoch.png', dpi=300,bbox_inches='tight')
plt.show()

NameError: name 'losses' is not defined

In [131]:
import pandas as pd 

# Analyze the results for one batch
test_batch = next(iter(test_loader))
pred = []
with torch.no_grad():
    test_batch.to(device)
    pred = model(test_batch.x.float(), test_batch.edge_index, test_batch.batch) 
    test1 = test_batch.y


In [132]:
test1.shape

torch.Size([3432, 429])

In [133]:
pred2 = pred.cpu().detach().numpy()
test2 = test1.cpu().detach().numpy()

In [134]:
test2

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [135]:
test3 = Bi.inverse_transform(test2)

In [136]:
test3

array(['P162', 'P866', 'P809', ..., 'P241', 'P53', 'P826'], dtype='<U5')

In [137]:
pred3 = Bi.inverse_transform(pred2)

In [138]:
pred3

array(['P162', 'P866', 'P809', ..., 'P241', 'P53', 'P826'], dtype='<U5')

In [147]:
Cls = pd.DataFrame([test3,pred3])
Cls

,0,1,2,3,4,5,6,7,8,9,...,3422,3423,3424,3425,3426,3427,3428,3429,3430,3431
0,P162,P866,P809,P295,P3,P235,P35,P805,P294,P302,...,P174,P107,P297,P992,P924,P794,P350,P241,P53,P826
1,P162,P866,P809,P295,P3,P235,P35,P805,P294,P302,...,P174,P31,P297,P992,P924,P794,P350,P241,P53,P826


In [148]:
Cls.to_csv('Clss_pred.csv', sep=',', encoding='utf-8', header='true')

In [139]:
from sklearn.metrics import accuracy_score
Report = accuracy_score(test3,pred3)    
Report

0.9484265734265734

In [140]:
#torch.save(model.state_dict(),'C:\\Users\\ADMIN\\OneDrive - Indian Institute of Technology Bombay\\Project Codes\\Leak location and size detection C-Town\\GNN classification\\GNNCtownLeak.pth')



In [141]:
df

NameError: name 'df' is not defined

In [ ]:
test_batch.y

In [ ]:
df["y_real"] = df["y_real"].apply(lambda row: row[0])
df["y_pred"] = df["y_pred"].apply(lambda row: row[0])
df.head(12)

In [ ]:
plt = sns.scatterplot(data=df, x="y_real", y="y_pred")
plt

In [ ]:
from sklearn.metrics import r2_score
                                     
R2 = r2_score(df.iloc[:,0], df.iloc[:,1])   

R2
                                     

In [ ]:
#torch.save(model.state_dict(),'C:\\Users\\ADMIN\\OneDrive - Indian Institute of Technology Bombay\\Project Codes\\CTown\\MultiplayerGCN\\Chebnet3Pressure.pth')

